# 📘 Legal Acts Formatting Notebook
*Using Hugging Face FLAN-T5 to generate legal questions from structured acts and save them in Alpaca format.*

In [ ]:
import json
import requests
import os
import time
from tqdm import tqdm  # For progress visualization

### 🔐 Hugging Face API Configuration
Set your Hugging Face API token here:

In [ ]:
# Replace with your Hugging Face API key
HF_API_TOKEN = "your_huggingface_api_key"
API_URL = "https://api-inference.huggingface.co/models/google/flan-t5-large"
HEADERS = {"Authorization": f"Bearer {HF_API_TOKEN}"}

### 🔁 Function: API Call with Retry Logic

In [ ]:
def call_api_with_retry(prompt, file_name, max_attempts=5, buffer_time=5, extra_parameters=None):
    data = {"inputs": prompt}
    if extra_parameters is not None:
        data["parameters"] = extra_parameters

    attempt = 0
    while attempt < max_attempts:
        response = requests.post(API_URL, headers=HEADERS, json=data)
        if response.status_code == 503:
            try:
                error_info = response.json()
                wait_time = error_info.get("estimated_time", 30) + buffer_time
            except Exception:
                wait_time = 30 + buffer_time
            print(f"🚨 Model loading. Waiting for {wait_time} seconds before retrying (attempt {attempt+1}/{max_attempts})...")
            time.sleep(wait_time)
            attempt += 1
        else:
            return response
    raise Exception(f"API request failed after {max_attempts} attempts at {file_name}")

### ❓ Function: Generate Legal Questions

In [ ]:
def generate_instructions(heading, content, file_name, num_questions=3):
    questions = []
    sampling_params = {"temperature": 0.9, "do_sample": True, "top_p": 0.95}

    for i in range(num_questions):
        prompt = (
            f"Generate a distinct, detailed, and comprehensive legal question that fully covers the entire scope of the section below. "
            f"Ensure that the question is specific to the section's content, is phrased as a question, and is unique compared to other questions about this section. "
            f"Question number: {i+1}\n\n"
            f"Section Title: {heading}\n"
            f"Content: {content}\n\n"
            f"Return only the question as a plain text string."
        )

        response = call_api_with_retry(prompt, file_name, extra_parameters=sampling_params)

        if response.status_code == 200:
            question = response.json()[0].get("generated_text", "").strip()
            if not question.endswith('?'):
                question = question.rstrip('.') + '?'
            questions.append(question)
        else:
            print(f"\n🚨 API Error: {response.status_code} - {response.text}")
            raise Exception(f"API request failed at {file_name}")
    return questions

### 📁 Define Input and Output Folders

In [ ]:
input_folder = "/content/legal_acts"
output_folder = "/content/formatted_acts"
os.makedirs(output_folder, exist_ok=True)
json_files = [f for f in os.listdir(input_folder) if f.endswith(".json")]

### ⚙️ Main Processing Loop

In [ ]:
try:
    for json_file in tqdm(json_files, desc="Processing Acts"):
        file_path = os.path.join(input_folder, json_file)
        output_file = os.path.join(output_folder, json_file.replace(".json", "_formatted.json"))

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        act_title = data.get("Act Title", json_file.replace(".json", "").replace("_", " ").title())
        output_data = []

        print(f"\n📜 Processing: {act_title}")
        print("-" * 80)

        act_definition = data.get("Act Definition", {}).get("0", "No definition available")
        output_data.append({
            "instruction": f"What is the purpose and legal scope of {act_title}?",
            "context": f"{act_title} - Act Definition",
            "response": act_definition
        })

        if "Sections" in data:
            sections = data["Sections"]
        elif "Chapters" in data:
            sections = {}
            for chapter_id, chapter_data in data["Chapters"].items():
                sections.update(chapter_data.get("Sections", {}))
        else:
            print(f"⚠ Skipping {json_file} (No 'Sections' or 'Chapters' key found)")
            continue

        for section, details in sections.items():
            heading = details.get("heading", f"Section {section}")
            paragraphs = []

            def extract_paragraphs(p):
                if isinstance(p, dict):
                    for key in p:
                        extract_paragraphs(p[key])
                else:
                    paragraphs.append(p)

            extract_paragraphs(details.get("paragraphs", {}))
            full_content = " ".join(paragraphs)

            questions = generate_instructions(heading, full_content, json_file, num_questions=3)
            section_context = f"{act_title} - Section {section}"

            for question in questions:
                output_data.append({
                    "instruction": question,
                    "context": section_context,
                    "response": full_content
                })

            print(f"✅ Processed Section {section} from {act_title}")

        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(output_data, f, indent=4)

        print(f"📂 Saved reformatted data to {output_file}")

    print("\n🎯 All files processed successfully!")

except Exception as e:
    print(f"\n🚨 Process terminated: {e}")
    print("⚠ All processed files have been saved in the output folder.")